# Certified systoles of the Bolza twins

## Objectives

This notebook has two objectives.

1. For each of 14 arithmetic hyperbolic surfaces, certify the length of the
   shortest closed geodesic, or systole.
2. For one genus-eight surface, use the systole certificate to prove
   $\lambda_1>1/4$ for the first nonzero Laplace eigenvalue.

Sections 1--4 establish the systole certificates by exact arithmetic and a
finite trace search. Section 5 applies the Selberg trace formula to the
genus-eight surface.

For a hyperbolic group element $\gamma$,

$$
\ell(\gamma)
=
2\operatorname{arccosh}\left(\frac{|\operatorname{tr}\gamma|}{2}\right).
$$

Since $\operatorname{arccosh}$ is increasing, the shortest closed geodesic is
determined by the smallest admissible absolute trace greater than $2$.

The geometric argument therefore has two parts:

- construct a group element with trace $T$;
- prove that no smaller admissible trace occurs.

The first gives an upper bound for the systole. The second gives the matching
lower bound.


## Setup

This notebook requiresn `python-flint` and
`matplotlib`. If needed, install them with


```python
!pip install python-flint matplotlib
```

The next cell creates the `results/` and `images/` folders.


In [ ]:
from pathlib import Path
import csv
import json

ROOT = Path.cwd()
RESULTS = ROOT / "results"
IMAGES = ROOT / "images"
RESULTS.mkdir(exist_ok=True)
IMAGES.mkdir(exist_ok=True)


## Arithmetic in $\mathbb Z[\sqrt2]$

We will be working in the quadratic integers $a+b\sqrt2$ with $a,b\in\mathbb Z$.  We store it as
the pair `(a,b)`.  Addition is pairwise, while

$$
(a+b\sqrt2)(c+d\sqrt2)=(ac+2bd)+(ad+bc)\sqrt2.
$$

The first code cell implements this representation and basic
arithmetic operations.


In [ ]:
# We represent a + b*sqrt(2) by the pair (a, b).
def qi(a=0, b=0):
    """Construct a quadratic integer in Z[sqrt(2)].

    Args:
        a: Coefficient of 1.
        b: Coefficient of sqrt(2).

    Returns:
        The pair (a, b) representing a + b*sqrt(2).
    """
    return (int(a), int(b))


def as_qi(x):
    """Convert a scalar to the quadratic-integer representation.

    Args:
        x: A quadratic-integer pair or an integer scalar.

    Returns:
        A quadratic-integer pair.
    """
    if isinstance(x, tuple):
        return x
    return qi(x, 0)


def q_add(x, y):
    """Add two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The sum x + y in pair representation.
    """
    y = as_qi(y)
    return qi(x[0] + y[0], x[1] + y[1])


def q_subtract(x, y):
    """Subtract two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The difference x - y in pair representation.
    """
    y = as_qi(y)
    return qi(x[0] - y[0], x[1] - y[1])


def q_negative(x):
    """Negate a quadratic integer.

    Args:
        x: Quadratic integer to negate.

    Returns:
        The additive inverse of x.
    """
    return qi(-x[0], -x[1])


def q_multiply(x, y):
    """Multiply two quadratic integers.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer.

    Returns:
        The product x*y in pair representation.
    """
    y = as_qi(y)
    a = x[0] * y[0] + 2 * x[1] * y[1]
    b = x[0] * y[1] + x[1] * y[0]
    return qi(a, b)

The field $\mathbb Q(\sqrt2)$ has two real embeddings:

$$\sigma_1(a+b\sqrt2)=a+b\sqrt2,\qquad\sigma_2(a+b\sqrt2)=a-b\sqrt2.$$

Conjugation exchanges them. Their product is a norm: $$N(a+b\sqrt2)=a^2-2b^2.$$

We'll see later that $\sigma_1$ describes hyperbolic motion while
$\sigma_2$ supplies the bound that makes the search finite.

In [ ]:
def q_conjugate(x):
    """Conjugates the real embedding of a quadratic integer.

    Args:
        x: Quadratic integer a + b*sqrt(2).

    Returns:
        The conjugate a - b*sqrt(2).
    """
    return qi(x[0], -x[1])


def q_norm(x):
    """Compute the field norm of a quadratic integer.

    Args:
        x: Quadratic integer a + b*sqrt(2).

    Returns:
        The integer a^2 - 2*b^2.
    """
    return x[0] * x[0] - 2 * x[1] * x[1]

We must decide inequalities such as $a+b\sqrt2>2$ without rounding $\sqrt2$.
If the rational and irrational terms have opposite signs, we compare their
squares.  Because $\sqrt2$ is irrational, equality cannot occur accidentally.


In [ ]:
def q_sign(x, embedding=1):
    """Determine the sign of a quadratic integer exactly.

    Args:
        x: Quadratic integer to evaluate.
        embedding: Real embedding, using 1 for sqrt(2) and -1 for -sqrt(2).

    Returns:
        -1, 0, or 1 according to the sign at the selected embedding.
    """
    a = x[0]
    b = embedding * x[1]

    if a == 0:
        return (b > 0) - (b < 0)
    if b == 0 or (a > 0) == (b > 0):
        return (a > 0) - (a < 0)

    # The two terms have opposite signs, so compare their squared magnitudes.
    difference_of_squares = a * a - 2 * b * b
    if difference_of_squares > 0:
        return (a > 0) - (a < 0)
    return (b > 0) - (b < 0)


def q_compare(x, y, embedding=1):
    """Compare two quadratic integers at a real embedding.

    Args:
        x: First quadratic integer.
        y: Second quadratic integer or integer scalar.
        embedding: Real embedding, using 1 for sqrt(2) and -1 for -sqrt(2).

    Returns:
        -1 if x < y, 0 if x == y, and 1 if x > y.
    """
    return q_sign(q_subtract(x, y), embedding)

For a principal ideal $(I)$, $I$ divides $x$ if $x/I\in\mathbb Z[\sqrt2]$. Rationalizing gives

$$\frac{x}{I}=\frac{x\overline I}{N(I)},$$

so divisibility reduces to two ordinary integer remainder checks.

In [ ]:
def q_divides(divisor, x):
    """Test divisibility in Z[sqrt(2)].

    Args:
        divisor: Proposed divisor.
        x: Quadratic integer to test.

    Returns:
        True if x/divisor lies in Z[sqrt(2)], and False otherwise.
    """
    norm = q_norm(divisor)
    if norm == 0:
        return x == ZERO
    numerator = q_multiply(x, q_conjugate(divisor))
    return numerator[0] % norm == 0 and numerator[1] % norm == 0

The next cell defines a function that converts a quadratic integer into a JSON format for computer certification.

In [ ]:
def q_json(x):
    """Convert a quadratic integer to a JSON-serializable dictionary.

    Args:
        x: Quadratic integer in pair representation.

    Returns:
        A dictionary containing the coefficients of 1 and sqrt(2).
    """
    return {"a": x[0], "b": x[1]}

Next we will store some commonly used constants for convenience.

In [ ]:
ZERO = qi(0, 0)
ONE = qi(1, 0)
SQRT2 = qi(0, 1)

### Testing
The following assertions test the norm, multiplication, and comparison operations on a simple example as a sanity check.

In [ ]:
q = qi(3, 2)
assert q_norm(q) == 1
assert q_multiply(q, q_conjugate(q)) == ONE
assert q_multiply(qi(1, 1), qi(1, 1)) == qi(3, 2)
assert q_compare(qi(9, 6), 2) > 0
print("basic Z[sqrt(2)] arithmetic: OK")

## The Bolza quaternion order

A hyperbolic surface can be written as a quotient
$\Gamma\backslash\mathbb H$, where $\Gamma$ is a discrete group of
orientation-preserving hyperbolic isometries. For the surfaces considered
here, the group is represented arithmetically inside a quaternion algebra.

Only the explicit coordinate model is needed for the computations. A
quaternion element is written as

$$x=c_0+c_1\alpha+c_2\beta+c_3\alpha\beta, \qquad c_i\in\mathbb Z[\sqrt2],$$

with a specified noncommutative multiplication law.

This representation provides exact group generators, reduced trace and norm,
and congruence conditions for the finite-index subgroups. A quaternion is
stored as four coordinates in $\mathbb Z[\sqrt2]$.


In [ ]:
# Represent a quaternion by four quadratic-integer coordinates.
# Coordinates are taken in the basis 1, alpha, beta, alpha*beta.
def bolza_scalar(x):
    """Embed a scalar in the quaternion basis.

    Args:
        x: Integer or quadratic integer to embed.

    Returns:
        A quaternion with x in the scalar coordinate.
    """
    return (as_qi(x), ZERO, ZERO, ZERO)


def bolza_add(x, y):
    """Add two quaternion elements.

    Args:
        x: First quaternion.
        y: Second quaternion.

    Returns:
        The coordinate-wise sum x + y.
    """
    answer = []
    for i in range(4):
        answer.append(q_add(x[i], y[i]))
    return tuple(answer)


def bolza_negative(x):
    """Negate a quaternion element.

    Args:
        x: Quaternion to negate.

    Returns:
        The additive inverse of x.
    """
    answer = []
    for coordinate in x:
        answer.append(q_negative(coordinate))
    return tuple(answer)


def bolza_subtract(x, y):
    """Subtract two quaternion elements.

    Args:
        x: First quaternion.
        y: Second quaternion.

    Returns:
        The difference x - y.
    """
    return bolza_add(x, bolza_negative(y))


def coordinate_vector(x0, x1, x2, x3):
    """Construct a quaternion coordinate vector.

    Args:
        x0: Scalar-basis coefficient.
        x1: Alpha-basis coefficient.
        x2: Beta-basis coefficient.
        x3: Alpha-beta-basis coefficient.

    Returns:
        A four-coordinate quaternion tuple.
    """
    return (as_qi(x0), as_qi(x1), as_qi(x2), as_qi(x3))


Multiplication is determined by

$$\alpha^2=-1+\alpha,\qquad \beta^2=-1+\beta,\qquad \beta\alpha=-(1+\sqrt2)+\alpha+\beta-\alpha\beta.$$

The algebra is noncommutative, so the order of the basis factors matters.  The
table below records the four coordinates of every basis product.


In [ ]:
# Entry [i][j] is the coordinate vector of basis_i*basis_j.
c = q_negative(q_add(ONE, SQRT2))
MULTIPLICATION_TABLE = (
    (
        coordinate_vector(1, 0, 0, 0),
        coordinate_vector(0, 1, 0, 0),
        coordinate_vector(0, 0, 1, 0),
        coordinate_vector(0, 0, 0, 1),
    ),
    (
        coordinate_vector(0, 1, 0, 0),
        coordinate_vector(-1, 1, 0, 0),
        coordinate_vector(0, 0, 0, 1),
        coordinate_vector(0, 0, -1, 1),
    ),
    (
        coordinate_vector(0, 0, 1, 0),
        coordinate_vector(c, 1, 1, -1),
        coordinate_vector(-1, 0, 1, 0),
        coordinate_vector(-1, 1, q_negative(SQRT2), 0),
    ),
    (
        coordinate_vector(0, 0, 0, 1),
        coordinate_vector(-1, q_negative(SQRT2), 1, 0),
        coordinate_vector(0, -1, 0, 1),
        coordinate_vector(-1, 0, 0, q_negative(SQRT2)),
    ),
)


We multiply two quaternion elements by distributing the four terms of each
factor, using the multiplication table for each basis product, and collecting
coordinates. The order of the basis factors must be preserved because the
algebra is noncommutative.


In [ ]:
def bolza_multiply(x, y):
    """Multiply two quaternion elements.

    Args:
        x: First quaternion.
        y: Second quaternion.

    Returns:
        The product x*y in the fixed quaternion basis.
    """
    answer = [ZERO, ZERO, ZERO, ZERO]
    # Distribute over the basis and accumulate the resulting coordinates.
    for i in range(4):
        for j in range(4):
            product_of_coordinates = q_multiply(x[i], y[j])
            basis_product = MULTIPLICATION_TABLE[i][j]
            for k in range(4):
                term = q_multiply(product_of_coordinates, basis_product[k])
                answer[k] = q_add(answer[k], term)
    return tuple(answer)


The quaternion algebra has analogues of matrix trace and determinant called
the reduced trace and reduced norm.

Let $x\mapsto\overline x$ denote the standard involution. Define

$$\operatorname{trd}(x)=x+\overline x, \qquad \operatorname{nrd}(x)=x\overline x.$$

These expressions are scalar. Under the real matrix representation used for
the hyperbolic action, reduced trace becomes ordinary matrix trace and reduced
norm becomes determinant.

Thus $\operatorname{nrd}(x)=1$ gives determinant one, and
$|\operatorname{trd}(x)|>2$ gives a hyperbolic element. The reduced trace
determines the corresponding geodesic length.


In [ ]:
def bolza_involution(x):
    """Apply the standard quaternion involution.

    Args:
        x: Quaternion element.

    Returns:
        The involution of x used to define reduced trace and norm.
    """
    first = q_add(q_add(q_add(x[0], x[1]), x[2]), q_negative(q_multiply(SQRT2, x[3])))
    return (first, q_negative(x[1]), q_negative(x[2]), q_negative(x[3]))


def bolza_trace(x):
    """Compute the reduced trace of a quaternion element.

    Args:
        x: Quaternion element.

    Returns:
        The scalar reduced trace as a quadratic integer.

    Raises:
        AssertionError: If x plus its involution is not scalar.
    """
    total = bolza_add(x, bolza_involution(x))
    assert total[1:] == (ZERO, ZERO, ZERO)
    return total[0]


def bolza_norm(x):
    """Compute the reduced norm of a quaternion element.

    Args:
        x: Quaternion element.

    Returns:
        The scalar reduced norm as a quadratic integer.

    Raises:
        AssertionError: If x times its involution is not scalar.
    """
    total = bolza_multiply(x, bolza_involution(x))
    assert total[1:] == (ZERO, ZERO, ZERO)
    return total[0]


The relevant subgroups are projective congruence subgroups. The condition is

$$x\equiv\pm1\pmod I.$$

Thus every coordinate of $x-1$, or every coordinate of $x+1$, is divisible by
the ideal generator $I$.

The signs $+1$ and $-1$ define the same projective class because $M$ and $-M$
represent the same element of $\mathrm{PSL}_2(\mathbb R)$.

The norm-one generators are denoted `a`, `b`, with inverses `A`, `B`. A word
such as `aBAb` specifies the corresponding product of generators from left to
right.


In [ ]:
ALPHA = (ZERO, ONE, ZERO, ZERO)
BETA = (ZERO, ZERO, ONE, ZERO)
GENERATORS = {
    "a": ALPHA,
    "A": bolza_involution(ALPHA),
    "b": BETA,
    "B": bolza_involution(BETA),
}


def evaluate_word(word):
    """Evaluate a word in the Bolza generators.

    Args:
        word: String in the generator alphabet a, A, b, B.

    Returns:
        The corresponding quaternion element.
    """
    answer = bolza_scalar(1)
    for letter in word.replace(" ", ""):
        answer = bolza_multiply(answer, GENERATORS[letter])
    return answer


In [ ]:
def bolza_json(x):
    """Convert a quaternion to JSON-serializable coordinates.

    Args:
        x: Quaternion element.

    Returns:
        A list of four quadratic-integer dictionaries.
    """
    answer = []
    for coordinate in x:
        answer.append(q_json(coordinate))
    return answer

### Testing

In [ ]:
def bolza_is_congruent_to_scalar(x, sign, ideal):
    """Test a quaternion congruence modulo an ideal.

    Args:
        x: Quaternion element.
        sign: Scalar representative, typically 1 or -1.
        ideal: Quadratic integer generating the ideal.

    Returns:
        True if every coordinate of x - sign is divisible by the ideal.
    """
    difference = bolza_subtract(x, bolza_scalar(sign))
    for coordinate in difference:
        if not q_divides(ideal, coordinate):
            return False
    return True

These assertions verify the multiplication table against the displayed
relations and check that the generators have norm one.

In [ ]:
assert bolza_multiply(ALPHA, ALPHA) == bolza_add(bolza_scalar(-1), ALPHA)
assert bolza_multiply(BETA, BETA) == bolza_add(bolza_scalar(-1), BETA)
assert bolza_norm(ALPHA) == ONE
assert bolza_norm(BETA) == ONE
assert bolza_norm(bolza_multiply(ALPHA, BETA)) == ONE
print("quaternion relations and norms: OK")


## Published words and systole upper bounds

For each ideal $I$, a published word gives an explicit element of the
projective congruence subgroup. If its trace is $T>2$, then it determines a
closed geodesic of length

$$2\operatorname{arccosh}(T/2).$$

Hence the word gives an upper bound for the systole. The next section supplies the
matching lower bound by excluding all smaller admissible traces.

Each record below stores the ideal $I$, the claimed trace, a word in the
generators, and the sign $+1$ or $-1$ of its congruence class.

The genus is

$$g=1+\frac{p(p^2-1)}{48}.$$


In [ ]:
# Each entry records one published witness.
# Lowercase letters denote alpha and beta; capitals denote their inverses.
TWINS = [
    {"prime": 7,  "ideal": qi(1, 2),  "trace": qi(7, 4),   "word": "aB" * 4, "sign": -1, "negate": False},
    {"prime": 7,  "ideal": qi(1, -2), "trace": qi(9, 6),   "word": "BABaba" * 2, "sign": -1, "negate": False},
    {"prime": 17, "ideal": qi(1, -3), "trace": qi(75, 53), "word": "abABAb" + "aBaBABAbabABaBab", "sign": 1, "negate": False},
    {"prime": 17, "ideal": qi(1, 3),  "trace": qi(79, 56), "word": "aBABabABab" * 2, "sign": -1, "negate": False},
    {"prime": 23, "ideal": qi(5, -1), "trace": qi(91, 65), "word": "BaBAbAbABaBabAbAba", "sign": 1, "negate": False},
    {"prime": 23, "ideal": qi(5, 1),  "trace": qi(119, 84), "word": "abaBaBaBabaaBAbaBaBaB", "sign": -1, "negate": False},
    {"prime": 31, "ideal": qi(9, 5),  "trace": qi(129, 90), "word": "BabABaBAba" * 2, "sign": -1, "negate": False},
    {"prime": 31, "ideal": qi(9, -5), "trace": qi(153, 109), "word": "babAbaBaBabABAbABaBaBA", "sign": 1, "negate": False},
    {"prime": 41, "ideal": qi(7, 2),  "trace": qi(281, 198), "word": "Ba" * 10, "sign": -1, "negate": False},
    {"prime": 41, "ideal": qi(7, -2), "trace": qi(295, 208), "word": "bAbAbABaBABabAbABabaBabA", "sign": -1, "negate": False},
    {"prime": 47, "ideal": qi(7, 1),  "trace": qi(499, 353), "word": "babAbaBAbAbAbABAbABabAbAbA", "sign": 1, "negate": False},
    {"prime": 47, "ideal": qi(7, -1), "trace": qi(529, 374), "word": "BabABabABaBa" * 2, "sign": -1, "negate": False},
    {"prime": 71, "ideal": qi(11, -5), "trace": qi(633, 449), "word": "babAbaBA" + "bABA" + "bAbABA" + "bABabA" + "babAbA", "sign": 1, "negate": False},
    {"prime": 71, "ideal": qi(11, 5), "trace": qi(951, 672), "word": "babAbaBAbAbAbaBABabaBAbAbAbA", "sign": -1, "negate": True},
]


def genus(datum):
    """Compute the genus associated with a witness datum.

    Args:
        datum: Witness dictionary containing the rational prime.

    Returns:
        The genus 1 + p(p^2 - 1)/48.
    """
    p = datum["prime"]
    return p * (p * p - 1) // 48 + 1


For one record, we evaluate the word and check that its reduced norm is one,
its reduced trace equals the claimed value, and it is congruent to the
recorded sign modulo $I$.

These checks certify that the proposed trace occurs in the group.


In [ ]:
def witness_element(datum):
    """Evaluate the quaternion element specified by a witness.

    Args:
        datum: Witness dictionary containing the word and sign convention.

    Returns:
        The corresponding quaternion element after the recorded negation.
    """
    element = evaluate_word(datum["word"])
    if datum["negate"]:
        element = bolza_negative(element)
    return element


def verify_witness(datum):
    """Verify the exact arithmetic conditions for a witness.

    Args:
        datum: Witness dictionary containing the ideal, trace, word, and sign.

    Returns:
        A JSON-serializable dictionary containing the witness and verification data.

    Raises:
        AssertionError: If the norm, trace, or congruence check fails.
    """
    element = witness_element(datum)
    checks = {
        "norm_one": bolza_norm(element) == ONE,
        "trace": bolza_trace(element) == datum["trace"],
        "ideal_congruence": bolza_is_congruent_to_scalar(element, datum["sign"], datum["ideal"]),
    }
    assert checks["norm_one"]
    assert checks["trace"]
    assert checks["ideal_congruence"]
    return {
        "prime": datum["prime"],
        "genus": genus(datum),
        "ideal": q_json(datum["ideal"]),
        "word": datum["word"],
        "word_length": len(datum["word"]),
        "word_negated": datum["negate"],
        "congruence_sign": datum["sign"],
        "trace": q_json(datum["trace"]),
        "coordinates": bolza_json(element),
        "checks": checks,
    }


### Testing
The same exact checks are applied to all 14 surfaces. The results are written
to `results/witnesses.json`.


In [ ]:
witness_rows = []
for datum in TWINS:
    witness_rows.append(verify_witness(datum))

(RESULTS / "witnesses.json").write_text(
    json.dumps({"witnesses": witness_rows}, indent=2) + "\n",
    encoding="utf-8",
)
print("verified", len(witness_rows), "witness words exactly")


## Finite trace enumeration

The surface group is infinite, but the congruence condition restricts its possible traces to a two-dimensional lattice.

Suppose $x\equiv\varepsilon\pmod I$, where $\varepsilon\in\{+1,-1\}$. Write $x=\varepsilon+Iy$. Since $\operatorname{nrd}(x)=1$, we have
$$1=1+\varepsilon I\,\operatorname{trd}(y)+I^2\operatorname{nrd}(y),$$
so $\varepsilon I\,\operatorname{trd}(y)=-I^2\operatorname{nrd}(y)$. Combining this with $\operatorname{trd}(x)=2\varepsilon+I\operatorname{trd}(y)$ gives
$$\operatorname{trd}(x)\in 2\varepsilon+I^2\mathbb Z[\sqrt2].$$
Thus every possible trace has the form
$$t=\pm2+I^2(m+n\sqrt2),\qquad m,n\in\mathbb Z.$$
The search for group elements is therefore reduced to a search over lattice points $(m,n)\in\mathbb Z^2$.

The two embeddings of $\mathbb Q(\sqrt2)$ are $\sigma_1(a+b\sqrt2)=a+b\sqrt2$ and $\sigma_2(a+b\sqrt2)=a-b\sqrt2$. The first embedding gives the hyperbolic representation. If $T$ is the trace of a known witness, any trace that could improve the corresponding systole upper bound must satisfy $2<\sigma_1(t)\le T$.

At the second embedding the quaternion algebra is definite, so norm-one elements satisfy $|\sigma_2(t)|\le2$. Hence every potentially shorter trace lies in the bounded region
$$2<\sigma_1(t)\le T,\qquad -2\le\sigma_2(t)\le2.$$

The admissible traces form a lattice in the $(\sigma_1,\sigma_2)$-plane, whose intersection with this bounded region is finite. The implementation uses a slightly larger integer box $|m|,|n|\le B$; it is sufficient to choose $B$ large enough that this box contains every lattice point satisfying the embedding bounds.

In [ ]:
from flint import arb, ctx


def coefficient_bound(ideal, upper_trace):
    """Compute a safe coefficient bound for trace enumeration.

    Args:
        ideal: Quadratic integer generating the congruence ideal.
        upper_trace: Trace upper bound supplied by the witness.

    Returns:
        An integer B such that the relevant lattice points lie in |m|, |n| <= B.
    """
    ideal_squared = q_multiply(ideal, ideal)
    ideal_norm = abs(q_norm(ideal))
    q_upper = abs(ideal_squared[0]) + 2 * abs(ideal_squared[1])
    trace_upper = abs(upper_trace[0]) + 2 * abs(upper_trace[1])
    numerator = (trace_upper + 6) * q_upper
    denominator = 2 * ideal_norm * ideal_norm
    return (numerator + denominator - 1) // denominator + 1


A candidate is admissible when it is hyperbolic at the first embedding and
lies between $-2$ and $2$ at the second.  These are exactly the three
inequalities in the next function.


In [ ]:
def admissible_trace(trace):
    """Test the necessary inequalities at both real embeddings.

    Args:
        trace: Candidate quadratic-integer trace.

    Returns:
        True if the trace satisfies the hyperbolic and conjugate-embedding bounds.
    """
    first_embedding_is_large = q_compare(trace, 2, 1) > 0
    second_embedding_is_at_least_minus_two = q_compare(trace, -2, -1) >= 0
    second_embedding_is_at_most_two = q_compare(trace, 2, -1) <= 0
    return (
        first_embedding_is_large
        and second_embedding_is_at_least_minus_two
        and second_embedding_is_at_most_two
    )


Both congruence signs and every pair $(m,n)$ in the certified search box are
checked.

For each lattice point,

$$t=\pm2+I^2(m+n\sqrt2)$$

is formed and tested against the two embedding conditions and the witness
trace bound.

These arithmetic conditions may include traces that are not realized by group
elements. This only enlarges the candidate set. Therefore, if the enlarged set
contains no trace below the witness trace, then no group element can have a
smaller admissible trace.


In [ ]:
def enumerate_minimum(ideal, upper_trace):
    """Enumerate the minimum admissible trace.

    Args:
        ideal: Quadratic integer generating the congruence ideal.
        upper_trace: Witness trace used as the enumeration cutoff.

    Returns:
        A tuple containing the minimum trace, its congruence sign, and the number
        of lattice points tested.

    Raises:
        AssertionError: If no admissible trace is found.
    """
    ideal_squared = q_multiply(ideal, ideal)
    bound = coefficient_bound(ideal, upper_trace)
    best = None
    best_sign = None
    tested = 0

    # Enumerate the two trace cosets +2 + I^2*z and -2 + I^2*z.
    # Enumerate every z = m + n*sqrt(2) in the certified coefficient box.
    for sign in [-1, 1]:
        for m in range(-bound, bound + 1):
            for n in range(-bound, bound + 1):
                tested = tested + 1
                lattice_point = q_multiply(ideal_squared, qi(m, n))
                trace = q_add(qi(2 * sign, 0), lattice_point)
                if not admissible_trace(trace):
                    continue
                if q_compare(trace, upper_trace) > 0:
                    continue
                if best is None or q_compare(trace, best) < 0:
                    best = trace
                    best_sign = sign

    assert best is not None
    return best, best_sign, tested


The length formula is

$$\ell = 2\operatorname{arccosh}\left(\frac{\sigma_1(t)}{2}\right).$$

Arb evaluates this expression with outward rounding and returns an interval
containing the exact length. A shorter decimal approximation is used only for
the displayed table.


In [ ]:
def length_interval(trace, digits=50):
    """Compute a rigorous interval for a hyperbolic length.

    Args:
        trace: Quadratic-integer trace.
        digits: Arb decimal precision.

    Returns:
        A two-element list containing lower and upper bounds for
        2*acosh(trace/2).
    """
    old_digits = ctx.dps
    ctx.dps = digits
    try:
        t = arb(trace[0]) + arb(trace[1]) * arb(2).sqrt()
        length = 2 * (t / 2).acosh()
        return [str(length.lower()), str(length.upper())]
    finally:
        ctx.dps = old_digits


def decimal_length(trace, digits=16):
    """Compute a decimal approximation of a hyperbolic length.

    Args:
        trace: Quadratic-integer trace.
        digits: Number of significant decimal digits to format.

    Returns:
        A formatted decimal string for 2*acosh(trace/2).
    """
    old_digits = ctx.dps
    ctx.dps = digits + 5
    try:
        t = arb(trace[0]) + arb(trace[1]) * arb(2).sqrt()
        length = 2 * (t / 2).acosh()
        return format(float(length.mid()), "." + str(digits) + "g")
    finally:
        ctx.dps = old_digits


The published word proves that the witness trace $T$ occurs. The finite
enumeration proves that no smaller admissible trace occurs. Their agreement
therefore certifies the systole.


In [ ]:
def build_systole_row(datum):
    """Build the certified systole record for one surface.

    Args:
        datum: Witness dictionary for the surface.

    Returns:
        A JSON-serializable dictionary containing the systole and its certificate.

    Raises:
        AssertionError: If the enumerated minimum does not match the witness data.
    """
    witness = witness_element(datum)
    minimum, sign, tested = enumerate_minimum(datum["ideal"], datum["trace"])
    assert minimum == datum["trace"]
    assert sign == datum["sign"]

    bound = coefficient_bound(datum["ideal"], datum["trace"])
    certificate = {
        "ideal": q_json(datum["ideal"]),
        "congruence_sign": sign,
        "search_bounds": {"m": [-bound, bound], "n": [-bound, bound]},
        "minimum_trace": q_json(minimum),
        "witness_word": datum["word"],
        "witness_coordinates": bolza_json(witness),
        "verified_length_interval": length_interval(minimum),
        "tested_lattice_points": tested,
    }

    return {
        "prime": datum["prime"],
        "genus": genus(datum),
        "ideal": q_json(datum["ideal"]),
        "ideal_norm": abs(q_norm(datum["ideal"])),
        "trace": q_json(minimum),
        "conjugate_trace": q_json(q_conjugate(minimum)),
        "systole_decimal": decimal_length(minimum, 18),
        "congruence_sign": sign,
        "word": datum["word"],
        "word_negated": datum["negate"],
        "certificate": certificate,
    }


The preceding calculation is applied to all 14 surfaces. Detailed
certificates are saved and a compact summary table is printed.


In [ ]:
systole_rows = []
for datum in TWINS:
    systole_rows.append(build_systole_row(datum))

with (RESULTS / "systoles.csv").open("w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file)
    writer.writerow(["prime", "genus", "ideal_a", "ideal_b", "trace_a", "trace_b", "systole"])
    for row in systole_rows:
        writer.writerow([
            row["prime"],
            row["genus"],
            row["ideal"]["a"],
            row["ideal"]["b"],
            row["trace"]["a"],
            row["trace"]["b"],
            row["systole_decimal"],
        ])

(RESULTS / "systoles.json").write_text(
    json.dumps({"surfaces": systole_rows}, indent=2) + "\n",
    encoding="utf-8",
)

for row in systole_rows:
    print(
        "p =", row["prime"],
        " genus =", row["genus"],
        " ideal =", row["ideal"],
        " trace =", row["trace"],
        " systole =", row["systole_decimal"],
    )


For $I=1-2\sqrt2$, both the finite search and the published witness give

$$t=9+6\sqrt2.$$

The record below also contains the search bounds, the number of lattice points
tested, and the rigorous interval for the geodesic length.


In [ ]:
genus8 = None
for row in systole_rows:
    if row["ideal"] == {"a": 1, "b": -2}:
        genus8 = row
        break

assert genus8 is not None
genus8


The preceding sections establish the geometric input for the spectral argument:

$$\text{explicit group element} \Longrightarrow \text{systole upper bound},$$

and

$$\text{trace congruence} + \text{two real embeddings} \Longrightarrow \text{finite exclusion of smaller traces}.$$

When the witness trace agrees with the smallest admissible trace from the
finite search, the systole is certified.

We next use this systole certificate in the Selberg trace formula to obtain
a lower bound for the first nonzero Laplace eigenvalue.


## Spectral gap

Write the Laplace eigenvalues as $0=\lambda_0<\lambda_1\le\lambda_2\le\cdots$ and set $\lambda_j=\frac14+r_j^2$. If $\lambda_j\le1/4$, then $r_j$ is purely imaginary and $r_j^2\in[-1/4,0]$.

Let $N_{[0,1/4]}$ denote the number of eigenvalues in $[0,1/4]$, counted with multiplicity. Since $\lambda_0=0$, we have $N_{[0,1/4]}\ge1$. Therefore, to prove
$$\lambda_1>\frac14,$$
it is enough to show
$$N_{[0,1/4]}<2.$$

For a suitable test function $h$ with Fourier transform $\widehat h$, the Selberg trace formula has the schematic form
$$\sum_j h(r_j)=\text{area term}+\sum_{\gamma}\bigl(\text{positive geometric weight}\bigr)\widehat h(\ell(\gamma)).$$
Only the sign of the geodesic term is needed here.

Let $s$ be smaller than the certified systole. If $\widehat h(x)\le0$ for all $x\ge s$, then every closed geodesic contributes a nonpositive term, since $\ell(\gamma)\ge\operatorname{sys}(X)>s$. Hence the entire geodesic sum is nonpositive and may be omitted when forming an upper bound for the spectral side.

Small eigenvalues $\lambda\in[0,1/4]$ correspond to $y=r^2\in[-1/4,0]$. Assume that $h(r)\ge m>0$ throughout this interval. Then every eigenvalue in $[0,1/4]$ contributes at least $m$ to the spectral sum.

The constant eigenfunction $\lambda_0=0$ contributes $h(i/2)$. With the normalization used below, the trace formula therefore gives
$$N_{[0,1/4]}\le 1+\frac{2(g-1)I-h(i/2)}{m},$$
where $I$ is an explicit integral depending on $h$.

The argument thus requires three ingredients: $\widehat h(x)\le0$ for $x\ge s$, so that the geodesic contribution is nonpositive whenever $s<\operatorname{sys}(X)$; $h(r)\ge m>0$ for $r^2\in[-1/4,0]$, so that each small eigenvalue contributes at least $m$; and a rigorous upper bound for $I$, which controls the remaining analytic term. If these estimates give
$$1+\frac{2(g-1)I-h(i/2)}{m}<2,$$
then $N_{[0,1/4]}<2$, excluding every nonzero eigenvalue in $[0,1/4]$.

The construction must control both $h$ and $\widehat h$. The functions $L_n^{-1/2}(x^2)e^{-x^2/2}$ are convenient because, in the Fourier normalization used here, the $n$th basis function is multiplied by $(-1)^n$ under Fourier transform.

A coefficient vector $(c_0,\ldots,c_{23})$ therefore determines the polynomials
$$u(y)=\sum_j c_jL_j^{-1/2}(y),\qquad v(y)=\sum_j(-1)^jc_jL_j^{-1/2}(y),$$
and hence
$$\widehat h(x)=u(x^2)e^{-x^2/2},\qquad h(r)=v(r^2)e^{-r^2/2}.$$

Since $e^{-x^2/2}>0$ on the real axis, the Gaussian factor does not affect signs. Thus the required sign conditions on $\widehat h$ and $h$ reduce to corresponding sign conditions on the polynomials $u$ and $v$.

The required sign pattern is imposed by prescribing roots of $u$ and $v$. On the geometric side, we impose $u(s^2)=0$. Additional roots are taken to be double, since a double root does not generically change the sign of a real polynomial; the same device is used for $v$.

The numerical root locations below were obtained in a preliminary search for a suitable test function. They are design parameters rather than distinguished constants. For the certificate, these locations are fixed and converted to exact rational numbers before any proof step is performed.

Prescribing roots does not by itself establish the required global signs, so four additional points must be certified.

First, an unprescribed real root could introduce an unwanted sign change. Sturm's theorem is therefore used to count all real roots on the relevant intervals.

Second, we need a positive lower bound for $v(y)e^{-y/2}$ on $y\in[-1/4,0]$. Its interior critical points are precisely the roots of
$$\frac{d}{dy}\left(v(y)e^{-y/2}\right)=0\quad\Longleftrightarrow\quad 2v'(y)-v(y)=0,$$
so Sturm root counts can be used to locate all possible interior minima.

Third, the trace-formula integral contains transcendental functions and extends to infinity. Its finite part is evaluated using rigorous interval arithmetic, while the remaining Gaussian tail is bounded explicitly.

Finally, the resulting trace-formula bound is itself evaluated as an interval. The certificate succeeds only if the upper endpoint of this interval is strictly smaller than $2$.

The preceding sections provide the exact systole. We choose $s<\operatorname{sys}(X)$ and construct a Fourier pair $(h,\widehat h)$ satisfying $\widehat h(x)\le0$ for all $x\ge s$. Since every closed geodesic has length greater than $s$, its contribution to the trace formula is nonpositive.

We also prove $h(r)\ge m>0$ whenever $r^2\in[-1/4,0]$. After rigorously bounding the trace-formula integral $I$, we obtain
$$N_{[0,1/4]}\le 1+\frac{2(g-1)I-h(i/2)}{m}<2.$$

Since $N_{[0,1/4]}$ is an integer and already counts the constant eigenfunction $\lambda_0=0$, it follows that $N_{[0,1/4]}=1$. Thus there are no nonzero eigenvalues in $[0,1/4]$, and therefore
$$\lambda_1>\frac14.$$

In [ ]:
from fractions import Fraction
from math import factorial
from flint import acb, arb, ctx, fmpq, fmpq_mat, fmpq_poly


GENUS = 8
SYSTOLE_PARAMETER = 5.481318

# Approximate root locations used in the numerical test-function design.
Z_FLOATS = [138.77578072264635]
W_FLOATS = [
    1.5317898207809217,
    5.081777473066674,
    8.702772446377038,
    12.83195711564963,
    19.80275236588154,
    26.938160595707124,
    35.58087288860836,
    47.92367335730668,
    64.66040321194924,
    160.10218577649115,
]


def rational_number(value):
    """Embed an integer or binary floating-point value in Q exactly.

    Args:
        value: Integer, float, or rational-compatible value.

    Returns:
        The corresponding FLINT rational number.
    """
    if isinstance(value, int):
        return fmpq(value)
    if isinstance(value, float):
        value = Fraction.from_float(value)
    return fmpq(value.numerator, value.denominator)


def fraction_json(value):
    """Convert a FLINT rational number to JSON form.

    Args:
        value: FLINT rational number.

    Returns:
        A dictionary containing its numerator and denominator.
    """
    return {"numerator": int(value.p), "denominator": int(value.q)}


The next cell constructs $L_n^{-1/2}$ exactly by recurrence. Alternating the
coefficient signs later produces the Fourier partner polynomial.


In [ ]:
def laguerre_basis(count):
    """Construct generalized Laguerre polynomials by recurrence.

    Args:
        count: Number of polynomials L_n^(-1/2) to construct.

    Returns:
        A list containing L_0^(-1/2) through L_(count-1)^(-1/2).
    """
    x = fmpq_poly([0, 1])
    polynomials = [fmpq_poly([1])]
    if count == 1:
        return polynomials

    polynomials.append(fmpq_poly([fmpq(1, 2), -1]))
    for n in range(1, count - 1):
        left = (fmpq(2 * n) + fmpq(1, 2) - x) * polynomials[n]
        right = (fmpq(n) - fmpq(1, 2)) * polynomials[n - 1]
        next_polynomial = (left - right) / (n + 1)
        polynomials.append(next_polynomial)
    return polynomials


The selected root locations are converted to exact rational values before
they enter the interpolation system.


In [ ]:
def exact_design_points():
    """Convert the test-function design data to exact rationals.

    Returns:
        A tuple containing the u-root points, v-root points, and s^2.
    """
    z = []
    for value in Z_FLOATS:
        z.append(rational_number(value))

    w = []
    for value in W_FLOATS:
        w.append(rational_number(value))

    # Convert the binary64 value of s^2 to its exact rational representation.
    s_squared = rational_number(float(SYSTOLE_PARAMETER) ** 2)
    return z, w, s_squared


For a polynomial $p$, the condition

$$p(a)=0$$

is one linear equation in its coefficients. A double root requires

$$p(a)=p'(a)=0.$$

The system imposes the prescribed roots of $u$ and $v$, together with the
normalization

$$v(-1/4)=1.$$

There are 24 linear conditions for 24 Laguerre coefficients.


In [ ]:
def interpolation_rows(basis, derivatives, s_squared, z, w):
    """Construct the rational interpolation system.

    Args:
        basis: Laguerre basis polynomials.
        derivatives: Derivatives of the basis polynomials.
        s_squared: Exact rational value of the systole parameter squared.
        z: Prescribed double-root locations for u.
        w: Prescribed double-root locations for v.

    Returns:
        The matrix rows encoding the root and normalization conditions.

    Raises:
        AssertionError: If the number of conditions does not match the basis size.
    """
    size = len(basis)
    rows = []

    # Impose u(s^2) = 0.
    rows.append([polynomial(s_squared) for polynomial in basis])

    # Impose a double root of u at each point in z.
    for point in z:
        rows.append([polynomial(point) for polynomial in basis])
        rows.append([polynomial(point) for polynomial in derivatives])

    # Alternating coefficient signs implement the Fourier transform on the basis.
    # Impose a double root of v at each point in w.
    for point in w:
        value_row = []
        derivative_row = []
        for j in range(size):
            value_row.append(((-1) ** j) * basis[j](point))
            derivative_row.append(((-1) ** j) * derivatives[j](point))
        rows.append(value_row)
        rows.append(derivative_row)

    # Normalize the scale by imposing v(-1/4) = 1.
    normalization_row = []
    for j in range(size):
        normalization_row.append(((-1) ** j) * basis[j](fmpq(-1, 4)))
    rows.append(normalization_row)

    assert len(rows) == size
    return rows


We now construct $u$ and $v$ from the interpolation conditions.

FLINT solves the interpolation system over $\mathbb Q$. The resulting
coefficients of

$$u=\sum_j c_jL_j^{-1/2}, \qquad v=\sum_j(-1)^jc_jL_j^{-1/2}$$

are exact rational numbers.


In [ ]:
def solve_interpolation_system(rows, basis):
    """Solve for the Fourier-pair polynomials u and v.

    Args:
        rows: Rows of the rational interpolation matrix.
        basis: Laguerre basis polynomials.

    Returns:
        A tuple containing the exact rational polynomials u and v.
    """
    size = len(basis)
    flat_matrix = []
    for row in rows:
        for entry in row:
            flat_matrix.append(entry)

    matrix = fmpq_mat(size, size, flat_matrix)
    right_hand_side = fmpq_mat(size, 1, [0] * (size - 1) + [1])
    solution = matrix.solve(right_hand_side, algorithm="dixon")

    u = fmpq_poly()
    v = fmpq_poly()
    for j in range(size):
        coefficient = solution[j, 0]
        u = u + coefficient * basis[j]
        v = v + ((-1) ** j) * coefficient * basis[j]
    return u, v


def build_fourier_pair():
    """Construct the exact Fourier-pair polynomials and design data.

    Returns:
        A tuple containing u, v, s^2, the u-root points, and the v-root points.
    """
    z, w, s_squared = exact_design_points()
    size = 2 + 2 * (len(z) + len(w))
    basis = laguerre_basis(size)

    derivatives = []
    for polynomial in basis:
        derivatives.append(polynomial.derivative())

    rows = interpolation_rows(basis, derivatives, s_squared, z, w)
    u, v = solve_interpolation_system(rows, basis)
    return u, v, s_squared, z, w


To exclude unprescribed real roots, the implementation uses Sturm's theorem.

For a polynomial $p$, the Sturm sequence begins

$$p,\quad p',\quad -\operatorname{rem}(p,p'),\ldots$$

and continues by the Euclidean algorithm. The computation is performed over
$\mathbb Q$.


In [ ]:
def sign_of_rational(value):
    """Return the sign of a rational number.

    Args:
        value: Rational value.

    Returns:
        -1, 0, or 1 according to the sign of value.
    """
    return (value > 0) - (value < 0)


def sturm_sequence(polynomial):
    """Construct the Sturm sequence of a rational polynomial.

    Args:
        polynomial: Rational polynomial.

    Returns:
        The signed Euclidean-remainder sequence used by Sturm's theorem.
    """
    sequence = [polynomial, polynomial.derivative()]
    if sequence[1] == 0:
        return sequence[:1]

    # Use the polynomial Euclidean algorithm with the Sturm sign convention.
    while sequence[-1] != 0:
        quotient, remainder = divmod(sequence[-2], sequence[-1])
        if remainder == 0:
            break
        sequence.append(-remainder)
    return sequence


Let $V(x)$ be the number of sign changes in a Sturm sequence evaluated at
$x$. The number of distinct roots in $(a,b)$ is

$$V(a)-V(b).$$

These counts certify that no unprescribed roots occur on the intervals where
the signs of $u$ and $v$ are used.


In [ ]:
def number_of_sign_changes(signs):
    """Count sign changes after removing zero entries.

    Args:
        signs: Sequence of values in {-1, 0, 1}.

    Returns:
        The number of sign changes in the nonzero subsequence.
    """
    nonzero_signs = []
    for sign in signs:
        if sign != 0:
            nonzero_signs.append(sign)

    changes = 0
    for i in range(len(nonzero_signs) - 1):
        if nonzero_signs[i] != nonzero_signs[i + 1]:
            changes = changes + 1
    return changes

def sturm_variation(sequence, point):
    """Evaluate the Sturm sign variation at a point.

    Args:
        sequence: Sturm sequence.
        point: Rational evaluation point, or None for positive infinity.

    Returns:
        The number of sign variations at the selected point.
    """
    signs = []
    if point is None:                 # the limit at positive infinity
        for polynomial in sequence:
            leading_coefficient = polynomial[polynomial.degree()]
            signs.append(sign_of_rational(leading_coefficient))
    else:
        for polynomial in sequence:
            signs.append(sign_of_rational(polynomial(point)))
    return number_of_sign_changes(signs)


def count_roots(polynomial, left, right=None):
    """Count distinct real roots in an interval using Sturm's theorem.

    Args:
        polynomial: Rational polynomial.
        left: Left endpoint of the interval.
        right: Right endpoint, or None for positive infinity.

    Returns:
        The number of distinct roots in the interval.

    Raises:
        AssertionError: If a finite endpoint is itself a root.
    """
    assert polynomial(left) != 0
    if right is not None:
        assert polynomial(right) != 0
    sequence = sturm_sequence(polynomial)
    return sturm_variation(sequence, left) - sturm_variation(sequence, right)


The trace-formula integral contains exponentials, hyperbolic functions, and an
improper integral. These quantities are evaluated with Arb interval
arithmetic.

An Arb ball $[m\pm r]$ is guaranteed to contain the exact value represented by
the computation. All rounding is outward. Thus an interval with upper endpoint
below $2$ certifies a strict bound below $2$.


In [ ]:
def arb_from_rational(value):
    """Embed an exact rational number in an Arb ball.

    Args:
        value: FLINT rational number.

    Returns:
        An Arb ball containing the exact rational value.
    """
    return arb(int(value.p)) / arb(int(value.q))


def evaluate_complex_polynomial(polynomial, value):
    """Evaluate a rational polynomial on a complex Arb ball.

    Args:
        polynomial: Rational polynomial.
        value: Complex Arb argument.

    Returns:
        The rigorous complex ball obtained by Horner evaluation.
    """
    answer = acb(0)
    coefficients = list(polynomial)
    coefficients.reverse()
    for coefficient in coefficients:
        answer = answer * value + arb_from_rational(coefficient)
    return answer


def evaluate_real_polynomial(polynomial, value):
    """Evaluate a rational polynomial on a real Arb ball.

    Args:
        polynomial: Rational polynomial.
        value: Real Arb argument.

    Returns:
        The rigorous real ball obtained by Horner evaluation.
    """
    answer = arb(0)
    coefficients = list(polynomial)
    coefficients.reverse()
    for coefficient in coefficients:
        answer = answer * value + arb_from_rational(coefficient)
    return answer


The improper integral is split as

$$[0,16]\cup[16,\infty).$$

On $[0,16]$, Arb performs rigorous numerical integration. The interval is
subdivided to improve the resulting enclosures.


In [ ]:
def finite_trace_integral(v, cutoff):
    """Rigorously integrate the trace-formula integrand to a cutoff.

    Args:
        v: Spectral polynomial in the test function.
        cutoff: Positive integer integration cutoff.

    Returns:
        An Arb interval containing the integral over [0, cutoff].
    """
    # Arb evaluates this callback during validated numerical integration.
    def integrand(x, analytic):
        """Evaluate the complex trace-formula integrand.

        Args:
            x: Complex Arb evaluation point.
            analytic: Analyticity flag supplied by the Arb integration interface.

        Returns:
            The complex Arb value of the integrand.
        """
        polynomial_value = evaluate_complex_polynomial(v, x * x)
        gaussian = (-(x * x) / 2).exp()
        return polynomial_value * gaussian * x * (acb.pi() * x).tanh()

    integral = arb(0)
    for left in range(0, cutoff, 2):
        piece = acb.integral(
            integrand,
            left,
            left + 2,
            rel_tol=arb("1e-45"),
            abs_tol=arb("1e-45"),
            deg_limit=80,
            eval_limit=200000,
            depth_limit=30,
            use_heap=True,
        )
        assert piece.is_finite()
        assert piece.imag.contains(0)
        integral = integral + piece.real
    return integral


For $r\ge16$,

$$|\tanh(\pi r)|\le1.$$

Each polynomial term is bounded against the Gaussian decay. Explicit Gaussian
moment estimates then give a rigorous upper bound for the integral over
$[16,\infty)$.


In [ ]:
def gaussian_tail_bound(v, cutoff):
    """Bound the trace-formula integral beyond a cutoff.

    Args:
        v: Spectral polynomial in the test function.
        cutoff: Positive tail cutoff.

    Returns:
        An Arb upper bound for the absolute Gaussian tail.
    """
    b = arb(cutoff)
    z = b * b / 2
    exponential = (-z).exp()
    tail = arb(0)
    coefficients = list(v)

    for k in range(len(coefficients)):
        incomplete_gamma = arb(0)
        term = arb(1)
        for j in range(k + 1):
            if j > 0:
                term *= z / j
            incomplete_gamma += term

        moment = (arb(2) ** k) * factorial(k) * exponential * incomplete_gamma
        tail += abs(arb_from_rational(coefficients[k])) * moment
    return tail


def trace_integral(v, cutoff=16):
    """Compute a rigorous enclosure of the full trace integral.

    Args:
        v: Spectral polynomial in the test function.
        cutoff: Positive integer separating finite integration from the tail bound.

    Returns:
        A tuple containing the full integral enclosure and the tail bound.
    """
    finite_part = finite_trace_integral(v, cutoff)
    tail = gaussian_tail_bound(v, cutoff)
    integral_with_tail = finite_part + arb(0, tail.upper())
    return integral_with_tail, tail


The following function checks the prescribed zeros, the intended double-root
multiplicities, the sign of the second derivative at each tangency, and the
required leading-coefficient signs. These checks use exact arithmetic.

These are local checks at the prescribed roots. Global sign control is
provided by the Sturm counts.


In [ ]:
def forced_polynomial_checks(u, v, s_squared, z, w):
    """Verify the prescribed roots and leading signs exactly.

    Args:
        u: Geometric-side rational polynomial.
        v: Spectral-side rational polynomial.
        s_squared: Exact threshold s^2.
        z: Prescribed double-root locations for u.
        w: Prescribed double-root locations for v.

    Returns:
        A dictionary of exact Boolean sign and multiplicity checks.

    Raises:
        AssertionError: If any prescribed algebraic condition fails.
    """
    du = u.derivative()
    dv = v.derivative()
    ddu = du.derivative()
    ddv = dv.derivative()

    checks = {
        "u_at_s": u(s_squared) == 0,
        "u_s_derivative_negative": du(s_squared) < 0,
        "u_double_roots": True,
        "v_double_roots": True,
        "u_leading_negative": u[u.degree()] < 0,
        "v_leading_positive": v[v.degree()] > 0,
    }

    for point in z:
        if not (u(point) == 0 and du(point) == 0 and ddu(point) < 0):
            checks["u_double_roots"] = False
    for point in w:
        if not (v(point) == 0 and dv(point) == 0 and ddv(point) > 0):
            checks["v_double_roots"] = False

    for check in checks.values():
        assert check
    return checks


Sturm counts exclude additional roots on all intervals where a sign condition
is required.

The roots of

$$2v'(y)-v(y)$$

on $[-1/4,0]$ are also counted. These are the possible interior critical
points of

$$v(y)e^{-y/2}.$$

Together with the endpoint evaluations, this gives a rigorous positive lower
bound $m$ for the spectral weight.


In [ ]:
def add_global_sign_checks(u, v, s_squared, z, w, checks):
    """Add Sturm-based global sign checks to a certificate.

    Args:
        u: Geometric-side rational polynomial.
        v: Spectral-side rational polynomial.
        s_squared: Exact threshold s^2.
        z: Prescribed double-root locations for u.
        w: Prescribed double-root locations for v.
        checks: Dictionary of previously verified algebraic checks.

    Returns:
        The updated checks dictionary with root counts and critical-point data.

    Raises:
        AssertionError: If an unexpected root or critical-point pattern is found.
    """
    factor = fmpq_poly([-s_squared, 1])
    u_after_s, remainder = divmod(u, factor)
    assert remainder == 0

    u_root_count = count_roots(u_after_s, s_squared)
    v_root_count = count_roots(v, fmpq(-1, 4))
    assert u_root_count == len(z)
    assert v_root_count == len(w)

    dv = v.derivative()
    derivative_of_weight = 2 * dv - v
    extrema = count_roots(derivative_of_weight, fmpq(-1, 4), fmpq(0))
    left_derivative = derivative_of_weight(fmpq(-1, 4))
    right_derivative = derivative_of_weight(0)
    assert extrema == 0 or (extrema == 1 and left_derivative > 0 > right_derivative)

    checks["distinct_u_roots_after_s"] = u_root_count
    checks["expected_u_roots_after_s"] = len(z)
    checks["distinct_v_roots_after_minus_quarter"] = v_root_count
    checks["expected_v_roots_after_minus_quarter"] = len(w)
    checks["spectral_weight_extrema_on_interval"] = extrema
    checks["spectral_weight_endpoint_derivative_signs"] = [
        sign_of_rational(left_derivative),
        sign_of_rational(right_derivative),
    ]
    return checks


The bound uses $s<\operatorname{sys}(X)$, the sign condition
$\widehat h(x)\le0$ for $x\ge s$, the lower bound $h(r)\ge m>0$
when $r^2\in[-1/4,0]$, and a rigorous enclosure for the integral $I$.

The next cell evaluates

$$N_{[0,1/4]} \le 1+\frac{2(g-1)I-h(i/2)}{m}$$

with outward-rounded interval arithmetic.


In [ ]:
def interval_bound(v, digits):
    """Evaluate the trace-formula eigenvalue-count bound rigorously.

    Args:
        v: Spectral polynomial in the test function.
        digits: Arb decimal precision.

    Returns:
        A dictionary containing interval enclosures for the integral, spectral
        minimum, and N_[0,1/4].

    Raises:
        AssertionError: If the spectral minimum is not positive or the certified
        upper bound is not strictly below 2.
    """
    old_digits = ctx.dps
    ctx.dps = digits
    try:
        integral, tail = trace_integral(v)
        h_left = evaluate_real_polynomial(v, arb(-1) / 4) * (arb(1) / 8).exp()
        h_right = evaluate_real_polynomial(v, arb(0))

        minimum_lower = min(h_left.lower(), h_right.lower())
        minimum_upper = min(h_left.upper(), h_right.upper())
        center = (minimum_lower + minimum_upper) / 2
        radius = (minimum_upper - minimum_lower) / 2
        minimum = arb(center, radius)
        assert minimum.lower() > 0

        numerator = 2 * (GENUS - 1) * integral - h_left
        n_small = 1 + numerator / minimum
        assert n_small.upper() < 2

        return {
            "integral_interval": [str(integral.lower()), str(integral.upper())],
            "integral_tail_upper": str(tail.upper()),
            "minimum_spectral_weight_interval": [str(minimum.lower()), str(minimum.upper())],
            "N_[0,1/4]_interval": [str(n_small.lower()), str(n_small.upper())],
            "strict_upper_bound": str(n_small.upper()),
        }
    finally:
        ctx.dps = old_digits


The following function runs the exact polynomial checks, Sturm counts, and
interval estimates in a fixed order and records their outputs.


In [ ]:
def verify_ramanujan_certificate(digits=60):
    """Construct and verify the genus-eight spectral certificate.

    Args:
        digits: Arb decimal precision.

    Returns:
        A JSON-serializable dictionary containing all exact and interval checks.
    """
    u, v, s_squared, z, w = build_fourier_pair()
    sign_checks = forced_polynomial_checks(u, v, s_squared, z, w)
    sign_checks = add_global_sign_checks(u, v, s_squared, z, w, sign_checks)
    intervals = interval_bound(v, digits)

    certificate = {
        "genus": GENUS,
        "systole_parameter_squared": fraction_json(s_squared),
        "systole_parameter_decimal": SYSTOLE_PARAMETER,
        "polynomial_degree": max(u.degree(), v.degree()),
        "sign_checks": sign_checks,
        "integral_interval": intervals["integral_interval"],
        "integral_tail_upper": intervals["integral_tail_upper"],
        "minimum_spectral_weight_interval": intervals["minimum_spectral_weight_interval"],
        "N_[0,1/4]_interval": intervals["N_[0,1/4]_interval"],
        "strict_upper_bound": intervals["strict_upper_bound"],
        "conclusion": "N_[0,1/4] < 2, hence lambda_1 > 1/4",
        "arithmetic": "exact fmpq coefficients and Arb balls",
    }
    return certificate


The final computation checks that the exact genus-eight systole is strictly
larger than the test-function threshold $s$ and evaluates the
trace-formula bound.

The decisive condition is that the upper endpoint of the resulting interval is
strictly less than $2$. It follows that

$$N_{[0,1/4]}=1$$

and hence

$$\lambda_1>\frac14.$$


In [ ]:
spectral = verify_ramanujan_certificate(60)

# Verify that the certified genus-eight systole exceeds the test-function threshold.
old_digits = ctx.dps
ctx.dps = 60
try:
    trace = arb(9) + arb(6) * arb(2).sqrt()
    exact_systole = 2 * (trace / 2).acosh()
    spectral["exact_surface_systole_interval"] = [
        str(exact_systole.lower()),
        str(exact_systole.upper()),
    ]
    spectral["systole_above_5.482"] = bool(exact_systole.lower() > arb("5.482"))
    spectral["systole_above_certificate_parameter"] = bool(
        exact_systole.lower() > arb(str(spectral["systole_parameter_decimal"]))
    )
finally:
    ctx.dps = old_digits

assert spectral["systole_above_5.482"]
assert spectral["systole_above_certificate_parameter"]
(RESULTS / "ramanujan_genus8.json").write_text(
    json.dumps(spectral, indent=2) + "\n",
    encoding="utf-8",
)
print("strict upper bound for N_[0,1/4]:", spectral["strict_upper_bound"])
print(spectral["conclusion"])


## Systole table plot

The plot summarizes the exact traces certified above. It is not used as
evidence in the proof.


In [ ]:
import matplotlib.pyplot as plt

# Plot one curve for each member of each twin pair.
by_prime = {}
for row in systole_rows:
    by_prime.setdefault(row["prime"], []).append(row)

fig, ax = plt.subplots(figsize=(7, 4))
for twin in (0, 1):
    points = [by_prime[p][twin] for p in sorted(by_prime)]
    ax.plot(
        [row["genus"] for row in points],
        [float(row["systole_decimal"]) for row in points],
        marker="o",
        label=f"twin {twin + 1}",
    )
ax.set_xscale("log")
ax.set_xlabel("genus (log scale)")
ax.set_ylabel("certified systole")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGES / "systoles_by_genus.png", dpi=180)
plt.show()


The second plot displays the geometric margin above the systole threshold and
the spectral margin below $N_{[0,1/4]}=2$.


In [ ]:
# Plot the geometric and spectral certification margins for the genus-eight surface.
exact_length = float(2 * (arb(9 + 6 * 2**0.5) / 2).acosh().mid())
n_upper = float(arb(spectral["strict_upper_bound"]).upper())
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].bar(["threshold", "exact systole"], [5.482, exact_length])
axes[0].set_ylim(5.4, 5.75)
axes[0].set_ylabel("length")
axes[1].bar(["certified upper bound", "target"], [n_upper, 2])
axes[1].set_ylim(1.9998, 2.00002)
axes[1].set_ylabel(r"$N_{[0,1/4]}$")
fig.tight_layout()
fig.savefig(IMAGES / "genus8_certificate.png", dpi=180)
plt.show()


## Output files and reproducibility checks

The notebook writes the witness checks, systole certificates, a CSV summary,
the genus-eight interval certificate, and two plots.

The arithmetic and geometric sections establish the systole certificates.
The spectral section gives the mathematical argument followed by the exact
and interval computations that certify its claims.

For further checks, modify a witness word and verify that an exact group
check fails. Inspect both real embeddings of a candidate trace and verify
the finite search bounds.

For the spectral calculation, replace a prescribed double root by a simple
root and inspect the resulting sign change, or perturb an interpolation
point and repeat the Sturm counts. Changing the integration cutoff allows
comparison of the finite integral and the certified Gaussian tail.

The systole proof uses exact word verification and exhaustive finite lattice
enumeration. The spectral proof uses exact polynomial arithmetic, Sturm root
counts, rigorous interval integration, an explicit tail bound, and the final
strict trace-formula inequality.
